# 오늘의집 집들이 — 사용자 맞춤 상품 조합 추천 모델

## 프로젝트 개요
오늘의집 히스토리 기반 추천의 한계(신규 회원 Cold Start 문제)를 보완하기 위해  
**사용자가 입력한 조건(거주유형·스타일·색조·예산 등) 기반 맞춤 상품 조합 추천 시스템**을 구축한다.

## 모델 파이프라인
```
원본 데이터 로드 + 전처리
      ↓
피처 테이블 생성 (원핫인코딩 + 수치형)
      ↓
클러스터링 기반 피처 선택
  → 하드 필터 변수 확정 (거주유형·평수·가족구성·시공방식·공사유형)
  → 소프트 필터 변수 확정 (색조·스타일·예산)
      ↓
KNN 추천 모델 학습 (하드 필터 → 그 안에서 KNN)
      ↓
Fallback 로직 (depth4 → depth3 → depth2 → depth1)
      ↓
인기도 점수 기반 최종 랭킹
      ↓
grid_zone 배치 데이터 집계
      ↓
최종 모델 저장 (recommend_model_final.pkl)
```

## 드롭박스 항목
| 구분 | 항목 | 근거 |
|---|---|---|
| 하드 필터 | 거주유형, 평수, 가족구성, 시공방식, 공사유형 | 클러스터링에서 압도적 중요도 확인 |
| 소프트 필터 | 색조, 스타일, 공간, 예산 | EDA에서 패턴 차이 검증 |
| 변경 범위 | 단일상품 / 특정공간 / 방전체 | - |
| 카테고리 | depth1 → depth2 → depth3 → depth4 | Fallback 로직 적용 |


---
## STEP 1. 데이터 로드 + 전처리 + 피처 테이블 생성

원본 데이터를 로드하고 모델링에 필요한 피처 테이블을 생성한다.

**전처리 내용**
- 리스트 컬럼 파싱 (styleList, familyList, constructions)
- 평수 구간화, 색조 그룹화, 지역 시/도 추출
- 인기도 점수 산출 (스크랩 40% + 좋아요 30% + 공유 20% + 댓글 10%)
- 태그 데이터 집계: 공간별 합산 예산, 공간 등장 여부, 카테고리 비율
- 범주형 원핫인코딩 + 수치형 정규화

**저장 파일**
- `model_feature_table.csv` — 모델 입력 피처 (169컬럼)
- `model_posts_meta.csv` — 게시글 메타 (인기도 + 예산)


---
## STEP 2. 피처 선택 — 클러스터링 기반

### 목적
어떤 변수가 드롭박스에 들어가야 하는지를 데이터로 검증한다.

### 방법론
1. **전체 피처로 클러스터링** → 시공 관련 변수가 군집을 지배함 확인
2. **시공 관련 피처 제거 후 재클러스터링** → 거주유형·평수·가족구성이 핵심 분리 변수임 확인
3. **하드 필터 변수도 제거 후 재클러스터링** → 색조·스타일 패턴 확인

### 결론
| 구분 | 변수 | 역할 |
|---|---|---|
| 하드 필터 | 거주유형, 평수, 가족구성, 시공방식, 공사유형 | 추천 풀 결정 — 클러스터링에서 압도적 중요도 |
| 소프트 필터 | 색조, 스타일, 예산 | KNN 유사도 계산 — EDA에서 패턴 차이 검증 |

> 색조/스타일이 클러스터링에서 중요도가 낮게 나온 이유:  
> 전체 데이터에서 무채색 71%, 내추럴 42%로 쏠려있어 분산이 작기 때문.  
> 하드 필터로 같은 조건끼리 묶은 후에는 색조/스타일 차이가 드러남 (EDA Q2/Q3 검증).


---
## STEP 3. KNN 추천 모델 학습 + 저장

### 모델 구조
```
하드 필터 (거주유형·평수·가족구성·시공방식·공사유형·예산)
      ↓ 조건에 맞는 게시글만 추출
KNN (색조·스타일·has_·budget_ 피처 기반 코사인 유사도)
      ↓ 하드 필터 결과 안에서 유사 게시글 K=50개 추출
변경범위별 상품 추출
      ↓ 단일상품 / 특정공간 / 방전체
Fallback (depth4 → depth3 → depth2 → depth1, 최소 30건)
      ↓
인기도 점수 기반 최종 랭킹
  최종점수 = 등장빈도(log, 50%) + 평균인기도(50%)
```

### 핵심 설계 원칙
- **하드 필터 먼저 → 그 안에서 KNN**: 이전 방식(전체 KNN → 교집합)은 후보가 20건으로 줄었으나, 개선 후 50건 안정적 확보
- **단계적 하드 필터 완화**: expertise 제거 → agent 제거 → 전체 순으로 완화
- **희소 조건 허용**: 데이터 한계로 인한 희소 조건(반셀프+부분공사 등)은 Fallback으로 처리

### 저장 파일
- `recommend_model_v2.pkl` — KNN 모델 패키지


---
## STEP 4. 추천 결과 검증

### 검증 목적
실제 상품 URL을 통해 추천 결과가 사용자 조건에 맞는지 직접 확인한다.

### 검증 시나리오
1. **아파트 30~40평 신혼부부 + 무채색 + 내추럴 + 소파** → 중고가 패브릭 소파 위주
2. **원룸 10평 싱글 + 웜톤 + 빈티지 + 소파 100만 이하** → 저가 빈티지 감성 소품
3. **아파트 30~40평 신혼부부 + 웜톤 + 빈티지 + 소파** → 가죽/레더 클래식 소파

**핵심 검증**: 같은 조건에서 색조/스타일만 바꿨을 때 다른 상품이 추천되는지 확인 ✅


---
## STEP 5. grid_zone 배치 데이터 집계 + 최종 모델 저장

### 목적
"방 전체 변경" 시나리오에서 추천된 상품을 사용자 방 사진의 **적절한 위치에 합성**하기 위한 배치 정보를 준비한다.

### grid_zone 구조
이미지를 9분할 (좌상단/상단/우상단/좌측/중앙/우측/좌하단/하단/우하단)

### 집계 내용
- **grid_zone 패턴**: 공간 × 구역별 자주 등장하는 카테고리 비율
- **대표 카테고리**: 공간 × 구역별 1위 카테고리
- **카테고리 위치**: 카테고리별 positionX/Y 중앙값

### 배치 로직 예시
```
거실 소파   → 우측  (X:0.485, Y:0.647)
거실 천장등 → 상단  (X:0.515, Y:0.193)
거실 러그   → 좌하단 (X:0.416, Y:0.854)
침실 커튼   → 상단  (X:0.492, Y:0.159)
```

### 최종 저장 파일
- `recommend_model_final.pkl` — 전체 모델 패키지 (KNN + 배치 데이터)
- `grid_zone_pattern.csv` — 공간×구역별 카테고리 비율
- `grid_zone_representative.csv` — 공간×구역별 대표 카테고리
- `category_position.csv` — 카테고리별 X/Y 위치 중앙값


---
## 모델링 완료

### 생성된 산출물
| 파일 | 설명 |
|---|---|
| `model_feature_table.csv` | 피처 테이블 (169컬럼) |
| `model_posts_meta.csv` | 게시글 메타 |
| `recommend_model_v2.pkl` | KNN 모델 패키지 |
| `recommend_model_final.pkl` | 최종 모델 (KNN + 배치 데이터) |
| `grid_zone_pattern.csv` | 공간×구역별 카테고리 패턴 |
| `grid_zone_representative.csv` | 공간×구역별 대표 카테고리 |
| `category_position.csv` | 카테고리별 위치 중앙값 |

### 다음 단계
대시보드 구현 — `recommend_model_final.pkl`을 불러다 추천 함수 호출만 하면 됨


In [1]:
# ══════════════════════════════════════════
# STEP 1. 데이터 로드 + 전처리 + 피처 테이블 생성
# ══════════════════════════════════════════

import pandas as pd
import numpy as np
import ast
import warnings
warnings.filterwarnings("ignore")

BASE = r"C:\Users\SAMSUNG\Desktop\ohouse_final"

# ── 1. 데이터 로드 ──
print("로딩 중...")
posts = pd.read_csv(f"{BASE}\\housewarming_posts_with_tone_urlinfo.csv")
tags  = pd.read_csv(f"{BASE}\\housewarming_product_tags_with_image.csv", low_memory=False)
space_df = pd.read_csv(f"{BASE}\\housewarming_posts_with_space_type.csv",
                       usecols=["contentId", "space_type"])

print(f"게시글: {posts.shape[0]:,}행 / 태그: {tags.shape[0]:,}행")
print(f"space_type 유효: {space_df['space_type'].notna().sum():,}건 / "
      f"결측: {space_df['space_type'].isna().sum():,}건")


# ── 2. 게시글 기본 전처리 ──
print("\n전처리 중...")

def parse_list_col(series):
    def _parse(x):
        try:
            return ast.literal_eval(x)
        except:
            return []
    return series.apply(_parse)

posts["styleList"]     = parse_list_col(posts["features_styleList"])
posts["familyList"]    = parse_list_col(posts["features_familyList"])
posts["constructions"] = parse_list_col(posts["features_constructions"])

posts["style_main"]        = posts["styleList"].apply(lambda x: x[0] if len(x) > 0 else None)
posts["family_main"]       = posts["familyList"].apply(lambda x: x[0] if len(x) > 0 else None)
posts["construction_main"] = posts["constructions"].apply(lambda x: x[0] if len(x) > 0 else None)

posts["area_num"] = posts["features_area"].str.extract(r"(\d+)").astype(float)

area_bins   = [0, 10, 20, 30, 40, 50, np.inf]
area_labels = ["~10평", "10~20평", "20~30평", "30~40평", "40~50평", "50평+"]
posts["area_range"] = pd.cut(posts["area_num"], bins=area_bins, labels=area_labels)

def group_tone(tone):
    if pd.isna(tone): return None
    tone = str(tone)
    if "Gray" in tone or "White" in tone or "Black" in tone: return "무채색"
    elif "Orange" in tone or "Yellow" in tone: return "웜톤"
    elif "Blue" in tone or "Sky" in tone: return "쿨톤"
    elif "Green" in tone: return "그린톤"
    elif "Red" in tone or "Pink" in tone: return "레드·핑크톤"
    else: return "기타"

posts["tone_group"]  = posts["primary_tone"].apply(group_tone)
posts["tone2_group"] = posts["secondary_tone"].apply(group_tone)
posts["sido"] = posts["features_region"].str.extract(
    r"^([\w]+(?:특별시|광역시|도|특별자치도|특별자치시)?)")

# ── 3. space_type 병합 ──
posts = posts.merge(space_df, on="contentId", how="left")
posts["space_type"] = posts["space_type"].fillna("unknown")
print(f"space_type 분포:\n{posts['space_type'].value_counts()}")

# ── 4. 인기도 점수 ──
for col in ["scrapCount", "likeCount", "shareCount", "commentAndReplyCount"]:
    posts[f"{col}_log"]  = np.log1p(posts[col])
    max_val = posts[f"{col}_log"].max()
    posts[f"{col}_norm"] = posts[f"{col}_log"] / max_val if max_val > 0 else 0

posts["popularity_score"] = (
    posts["scrapCount_norm"]           * 0.40 +
    posts["likeCount_norm"]            * 0.30 +
    posts["shareCount_norm"]           * 0.20 +
    posts["commentAndReplyCount_norm"] * 0.10
)
posts["popularity_score"] = (
    posts["popularity_score"] / posts["popularity_score"].max() * 100
).round(2)

print(f"게시글 전처리 완료: {posts.shape[1]}컬럼")


# ── 5. 태그 데이터 집계 ──
print("태그 데이터 집계 중...")

tags_valid = tags.dropna(subset=["productId", "sellingPrice", "ohou_category_depth1"])
tags_valid = tags_valid[tags_valid["sellingPrice"] > 0]
tags_dedup = tags_valid.drop_duplicates(subset=["contentId", "productId"])

VALID_PLACES = [
    "거실", "주방", "침실", "아이방", "원룸",
    "서재&작업실", "욕실", "베란다", "드레스룸",
    "사무공간", "현관", "복도", "가구&소품"
]

tags_place = tags_dedup[tags_dedup["place_label"].isin(VALID_PLACES)]
tags_place_dedup = tags_place.drop_duplicates(
    subset=["contentId", "place_label", "productId"])

place_budget = (
    tags_place_dedup.groupby(["contentId", "place_label"])["sellingPrice"]
    .sum().reset_index()
)

for place in VALID_PLACES:
    sub = place_budget[place_budget["place_label"] == place][["contentId", "sellingPrice"]]
    sub = sub.rename(columns={"sellingPrice": f"budget_{place}"})
    posts = posts.merge(sub, on="contentId", how="left")
    place_posts = set(tags_place[tags_place["place_label"] == place]["contentId"].unique())
    posts[f"has_{place}"] = posts["contentId"].isin(place_posts).astype(int)

print(f"태그 집계 완료: {posts.shape[1]}컬럼")


# ── 6. 피처 테이블 구성 ──
print("\n피처 테이블 구성 중...")

cat_cols = [
    "features_residence", "area_range", "features_agent",
    "features_expertise", "style_main", "family_main",
    "construction_main", "tone_group", "tone2_group",
    "sido", "writer_userableType",
    "space_type",  # ← 추가
]

for col in cat_cols:
    if hasattr(posts[col], 'cat'):
        posts[col] = posts[col].astype(str)
    posts[col] = posts[col].fillna("unknown").astype(str)

posts_encoded = pd.get_dummies(posts[cat_cols], prefix=cat_cols)

num_cols = (
    ["area_num", "primary_ratio", "secondary_ratio", "popularity_score"] +
    [c for c in posts.columns if c.startswith("budget_")] +
    [c for c in posts.columns if c.startswith("has_")]
)

posts_num = posts[num_cols].fillna(0)

feature_df = pd.concat([
    posts[["contentId"]].reset_index(drop=True),
    posts_encoded.reset_index(drop=True),
    posts_num.reset_index(drop=True)
], axis=1)

print(f"피처 테이블: {feature_df.shape[0]:,}행 × {feature_df.shape[1]}컬럼")
print(f"  - 범주형 원핫: {posts_encoded.shape[1]}개")
print(f"  - 수치형: {posts_num.shape[1]}개")

feature_df.to_csv(f"{BASE}\\model_feature_table.csv", index=False, encoding="utf-8-sig")
posts[["contentId", "popularity_score"] +
      [c for c in posts.columns if c.startswith("budget_")]
     ].to_csv(f"{BASE}\\model_posts_meta.csv", index=False, encoding="utf-8-sig")

print("\n저장 완료: model_feature_table.csv / model_posts_meta.csv")

로딩 중...
게시글: 8,000행 / 태그: 1,941,343행
space_type 유효: 6,558건 / 결측: 1,442건

전처리 중...
space_type 분포:
space_type
TR         1898
RT         1500
unknown    1442
TH          847
TF          417
HT          348
RH          333
HR          324
RF          177
TT          172
FR          160
FT          153
RR           86
FH           55
HF           46
HH           26
FF           16
Name: count, dtype: int64
게시글 전처리 완료: 47컬럼
태그 데이터 집계 중...
태그 집계 완료: 73컬럼

피처 테이블 구성 중...
피처 테이블: 8,000행 × 138컬럼
  - 범주형 원핫: 107개
  - 수치형: 30개

저장 완료: model_feature_table.csv / model_posts_meta.csv


In [2]:
# ══════════════════════════════════════════
# STEP 3. KNN 추천 모델 학습 + 저장
# ══════════════════════════════════════════

import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
import ast
import pickle
import warnings
warnings.filterwarnings("ignore")

BASE = r"C:\Users\SAMSUNG\Desktop\ohouse_final"

# ── KNN 피처 정의 ──
KNN_CAT_COLS = [
    "style_main",
    "tone_group",
    "tone2_group",
    "space_type",   # ← 추가
]
KNN_NUM_COLS = (
    ["primary_ratio"] +
    [f"budget_{p}" for p in VALID_PLACES] +
    [f"has_{p}" for p in VALID_PLACES]
)

def build_feature_matrix(df):
    df = df.copy()
    for col in KNN_CAT_COLS:
        df[col] = df[col].fillna("unknown").astype(str)
    encoded = pd.get_dummies(df[KNN_CAT_COLS], prefix=KNN_CAT_COLS)
    num     = df[KNN_NUM_COLS].fillna(0)
    return pd.concat([
        df[["contentId"]].reset_index(drop=True),
        encoded.reset_index(drop=True),
        num.reset_index(drop=True)
    ], axis=1), list(encoded.columns)

X_full, encoded_cols = build_feature_matrix(posts)
feature_cols = encoded_cols + KNN_NUM_COLS

print(f"KNN 피처 수: {len(feature_cols)}개")

scaler = StandardScaler()
scaler.fit(X_full[feature_cols].values)
print("스케일러 학습 완료")


# ── 추천 함수 ──
def recommend(
    # 하드 필터
    residence=None,
    area_range=None,
    family=None,
    agent=None,
    expertise=None,
    # 소프트 필터
    tone=None,
    style=None,
    space=None,
    budget=None,
    space_type=None,    # ← 추가
    # 변경 범위
    change_type="방전체",
    # 카테고리
    depth1=None,
    depth2=None,
    depth3=None,
    depth4=None,
    # 파라미터
    top_n=10,
    K=50,
    min_posts=10
):
    # ── Step 1. 하드 필터 ──
    filtered = posts.copy()
    if residence: filtered = filtered[filtered["features_residence"] == residence]
    if area_range: filtered = filtered[filtered["area_range"] == area_range]
    if family:    filtered = filtered[filtered["family_main"] == family]
    if agent:     filtered = filtered[filtered["features_agent"] == agent]
    if expertise: filtered = filtered[filtered["features_expertise"] == expertise]
    if space_type: filtered = filtered[filtered["space_type"] == space_type]  # ← 추가
    if budget and space:
        budget_col = f"budget_{space}"
        if budget_col in filtered.columns:
            filtered = filtered[
                filtered[budget_col].isna() | (filtered[budget_col] <= budget)
            ]

    # ── 단계적 완화 ──
    if len(filtered) < min_posts:
        print(f"  ⚠️ expertise 완화 ({len(filtered)}건 → ", end="")
        filtered = posts.copy()
        if residence: filtered = filtered[filtered["features_residence"] == residence]
        if area_range: filtered = filtered[filtered["area_range"] == area_range]
        if family:    filtered = filtered[filtered["family_main"] == family]
        if agent:     filtered = filtered[filtered["features_agent"] == agent]
        if space_type: filtered = filtered[filtered["space_type"] == space_type]
        print(f"{len(filtered)}건)")

    if len(filtered) < min_posts:
        print(f"  ⚠️ agent 완화 ({len(filtered)}건 → ", end="")
        filtered = posts.copy()
        if residence: filtered = filtered[filtered["features_residence"] == residence]
        if area_range: filtered = filtered[filtered["area_range"] == area_range]
        if family:    filtered = filtered[filtered["family_main"] == family]
        if space_type: filtered = filtered[filtered["space_type"] == space_type]
        print(f"{len(filtered)}건)")

    if len(filtered) < min_posts:
        print(f"  ⚠️ space_type 완화 ({len(filtered)}건 → ", end="")
        filtered = posts.copy()
        if residence: filtered = filtered[filtered["features_residence"] == residence]
        if area_range: filtered = filtered[filtered["area_range"] == area_range]
        if family:    filtered = filtered[filtered["family_main"] == family]
        print(f"{len(filtered)}건)")

    if len(filtered) < min_posts:
        print(f"  ⚠️ 전체 사용")
        filtered = posts.copy()

    print(f"  하드 필터 후: {len(filtered):,}건")

    # ── Step 2. KNN ──
    X_filtered, _ = build_feature_matrix(filtered)
    for col in feature_cols:
        if col not in X_filtered.columns:
            X_filtered[col] = 0
    X_filtered_scaled = scaler.transform(X_filtered[feature_cols].values)

    user_row = filtered.iloc[[0]].copy()
    if style:      user_row["style_main"] = style
    if tone:       user_row["tone_group"] = tone
    if space_type: user_row["space_type"] = space_type
    if budget and space: user_row[f"budget_{space}"] = budget
    if space:      user_row[f"has_{space}"] = 1

    X_user, _ = build_feature_matrix(user_row)
    for col in feature_cols:
        if col not in X_user.columns:
            X_user[col] = 0
    user_scaled = scaler.transform(X_user[feature_cols].values)

    k_actual = min(K, len(filtered))
    knn = NearestNeighbors(n_neighbors=k_actual, metric="cosine",
                           algorithm="brute", n_jobs=-1)
    knn.fit(X_filtered_scaled)
    _, indices = knn.kneighbors(user_scaled)
    candidate_ids = filtered.iloc[indices[0]]["contentId"].values
    print(f"  KNN 후보: {len(candidate_ids):,}건")

    # ── Step 3. 상품 추출 + Fallback ──
    candidate_tags = tags_valid[tags_valid["contentId"].isin(candidate_ids)].copy()

    if change_type in ["특정공간", "단일상품"] and space:
        candidate_tags = candidate_tags[candidate_tags["place_label"] == space]

    def filter_with_fallback(df, d1, d2, d3, d4, min_count=30):
        for depth_col, val in [
            ("ohou_category_depth4", d4),
            ("ohou_category_depth3", d3),
            ("ohou_category_depth2", d2),
            ("ohou_category_depth1", d1),
        ]:
            if val:
                sub = df[df[depth_col] == val]
                if len(sub) >= min_count:
                    print(f"  Fallback: {depth_col} 사용 ({len(sub)}건)")
                    return sub
        print(f"  Fallback: 카테고리 필터 없음 ({len(df)}건)")
        return df

    if change_type in ["단일상품", "특정공간"] and any([depth1, depth2, depth3, depth4]):
        candidate_tags = filter_with_fallback(
            candidate_tags, depth1, depth2, depth3, depth4
        )

    # ── Step 4. 인기도 랭킹 ──
    candidate_tags = candidate_tags.merge(
        posts[["contentId", "popularity_score"]], on="contentId", how="left"
    )
    product_score = (
        candidate_tags.groupby([
            "productId", "productName", "brand", "sellingPrice",
            "ohou_category_depth1", "ohou_category_depth2",
            "ohou_category_depth3", "originalImageUrl", "productUrl"
        ])
        .agg(등장수=("contentId","count"), 평균인기도=("popularity_score","mean"))
        .reset_index()
    )
    product_score = product_score[product_score["등장수"] >= 2]
    if len(product_score) == 0:
        product_score = (
            candidate_tags.groupby([
                "productId", "productName", "brand", "sellingPrice",
                "ohou_category_depth1", "ohou_category_depth2",
                "ohou_category_depth3", "originalImageUrl", "productUrl"
            ])
            .agg(등장수=("contentId","count"), 평균인기도=("popularity_score","mean"))
            .reset_index()
        )

    if len(product_score) == 0:
        print("  ❌ 추천 결과 없음")
        return pd.DataFrame()

    max_log = np.log1p(product_score["등장수"]).max()
    max_pop = product_score["평균인기도"].max()
    product_score["최종점수"] = (
        np.log1p(product_score["등장수"]) / max_log * 0.5 +
        product_score["평균인기도"] / max_pop * 0.5
        if max_log > 0 and max_pop > 0 else 0
    )

    result = product_score.sort_values("최종점수", ascending=False).head(top_n)

    print(f"\n  추천 결과 ({len(result)}개):")
    print(f"  {'상품명':<40} {'브랜드':<12} {'가격':>10} {'점수':>6}")
    print("  " + "-" * 72)
    for _, row in result.iterrows():
        print(f"  {str(row['productName'])[:40]:<40} "
              f"{str(row['brand'])[:12]:<12} "
              f"{row['sellingPrice']:>9,.0f}원 "
              f"{row['최종점수']:>5.3f}")

    return result

KNN 피처 수: 66개
스케일러 학습 완료


---
## STEP 4-A. 인기도 가중치 재설계 — 엔트로피 + 상관관계 기반

### 목적
기존 고정 가중치(스크랩 40% + 좋아요 30% + 공유 20% + 댓글 10%)를
데이터 기반으로 재산출한다.

### 방법
- **엔트로피**: 정보량 많은 지표에 높은 가중치
- **상관관계**: 중복 신호가 많은 지표에 낮은 가중치
- `가중치 = 엔트로피 × (1 - 평균상관계수)` 후 정규화

### 결론
| 지표 | 가중치 | 해석 |
|---|---|---|
| scrapRate | 31.2% | 조회 대비 저장률 (전환율, 구매 의향) |
| commentAndReplyCount | 24.9% | 깊은 관심도 |
| shareCount | 18.4% | 바이럴 지수 |
| likeCount | 13.0% | 가벼운 반응 |
| scrapCount | 12.5% | 절대 저장량 |

> 이 가중치는 STEP 1 최종 전처리 코드(WEIGHTS_FINAL)에 하드코딩되어 반영됨


In [6]:
# ══════════════════════════════════════════
# 인기도 점수 재설계 — 엔트로피 + 상관관계 기반
# ══════════════════════════════════════════

import pandas as pd
import numpy as np
from scipy.stats import entropy
import ast
import warnings
warnings.filterwarnings("ignore")

BASE = r"C:\Users\SAMSUNG\Desktop\ohouse_final"

posts = pd.read_csv(f"{BASE}\\housewarming_posts_with_tone_urlinfo.csv")

# ── 분석 대상 지표 ──
METRICS = ["viewCount", "likeCount", "scrapCount", "shareCount", "commentAndReplyCount"]

# ══════════════════════════════════════════
print("=" * 60)
print("STEP 1. 기존 인기도 점수 계산 (비교용)")
print("=" * 60)

for col in ["scrapCount", "likeCount", "shareCount", "commentAndReplyCount"]:
    posts[f"{col}_log_old"]  = np.log1p(posts[col])
    max_val = posts[f"{col}_log_old"].max()
    posts[f"{col}_norm_old"] = posts[f"{col}_log_old"] / max_val if max_val > 0 else 0

posts["popularity_score"] = (
    posts["scrapCount_norm_old"]           * 0.40 +
    posts["likeCount_norm_old"]            * 0.30 +
    posts["shareCount_norm_old"]           * 0.20 +
    posts["commentAndReplyCount_norm_old"] * 0.10
)
posts["popularity_score"] = (
    posts["popularity_score"] / posts["popularity_score"].max() * 100
).round(2)

print(f"기존 인기도 점수 계산 완료")
print(posts["popularity_score"].describe().round(2))


# ══════════════════════════════════════════
print("\n" + "=" * 60)
print("STEP 2. 상관관계 분석 — 중복 신호 확인")
print("=" * 60)

corr_matrix = posts[METRICS].corr()
print(f"\n지표 간 상관계수:")
print(corr_matrix.round(3))

# 각 지표의 평균 상관계수 (자기 자신 제외)
mean_corr = {}
for m in METRICS:
    others = [x for x in METRICS if x != m]
    mean_corr[m] = corr_matrix[m][others].abs().mean()

print(f"\n지표별 평균 상관계수 (높을수록 중복 신호):")
for m, v in sorted(mean_corr.items(), key=lambda x: x[1], reverse=True):
    print(f"  {m:<35}: {v:.3f}")


# ══════════════════════════════════════════
print("\n" + "=" * 60)
print("STEP 3. 엔트로피 분석 — 정보량 계산")
print("=" * 60)

entropies = {}
for m in METRICS:
    log_vals = np.log1p(posts[m].fillna(0).values)
    counts, _ = np.histogram(log_vals, bins=50)
    counts = counts + 1
    prob = counts / counts.sum()
    entropies[m] = entropy(prob)

print(f"\n지표별 엔트로피 (높을수록 정보량 많음):")
for m, v in sorted(entropies.items(), key=lambda x: x[1], reverse=True):
    print(f"  {m:<35}: {v:.3f}")


# ══════════════════════════════════════════
print("\n" + "=" * 60)
print("STEP 4. 가중치 계산 (엔트로피 × 독립성)")
print("=" * 60)

# viewCount → 스크랩 전환율로 변환
posts["scrapRate"] = posts["scrapCount"] / (posts["viewCount"] + 1)

# viewCount 제외하고 scrapRate 추가
METRICS_FINAL = ["scrapRate", "scrapCount", "likeCount", "shareCount", "commentAndReplyCount"]

# 로그 정규화
for m in METRICS_FINAL:
    posts[f"{m}_log"]  = np.log1p(posts[m].fillna(0))
    max_val = posts[f"{m}_log"].max()
    posts[f"{m}_norm"] = posts[f"{m}_log"] / max_val if max_val > 0 else 0

# 엔트로피 계산
entropies_final = {}
for m in METRICS_FINAL:
    log_vals = posts[f"{m}_log"].values
    counts, _ = np.histogram(log_vals, bins=50)
    counts = counts + 1
    prob = counts / counts.sum()
    entropies_final[m] = entropy(prob)

# 상관관계 계산
corr_final = posts[[f"{m}_norm" for m in METRICS_FINAL]].corr()
mean_corr_final = {}
for m in METRICS_FINAL:
    col = f"{m}_norm"
    others = [f"{x}_norm" for x in METRICS_FINAL if x != m]
    mean_corr_final[m] = corr_final[col][others].abs().mean()

# 가중치 = 엔트로피 × (1 - 평균상관계수)
raw_weights_final = {}
for m in METRICS_FINAL:
    raw_weights_final[m] = entropies_final[m] * (1 - mean_corr_final[m])

total_final = sum(raw_weights_final.values())
weights_final = {m: v / total_final for m, v in raw_weights_final.items()}

print(f"\n최종 가중치:")
for m, v in sorted(weights_final.items(), key=lambda x: x[1], reverse=True):
    print(f"  {m:<35}: {v:.3f} ({v*100:.1f}%)")


# ══════════════════════════════════════════
print("\n" + "=" * 60)
print("STEP 5. 새 인기도 점수 계산")
print("=" * 60)

posts["popularity_score_new"] = sum(
    posts[f"{m}_norm"] * weights_final[m] for m in METRICS_FINAL
)
posts["popularity_score_new"] = (
    posts["popularity_score_new"] / posts["popularity_score_new"].max() * 100
).round(2)

print(f"\n기존 인기도 점수 분포:")
print(posts["popularity_score"].describe().round(2))

print(f"\n새 인기도 점수 분포:")
print(posts["popularity_score_new"].describe().round(2))

# 상위 10개 게시글 비교
print(f"\n[상위 10개 게시글 비교]")
print(f"  {'기존 contentId':>15} {'기존점수':>8}  |  {'새 contentId':>15} {'새점수':>8}")
print("  " + "-" * 60)

old_top = posts.nlargest(10, "popularity_score")[
    ["contentId", "popularity_score"]].reset_index(drop=True)
new_top = posts.nlargest(10, "popularity_score_new")[
    ["contentId", "popularity_score_new"]].reset_index(drop=True)

for i in range(10):
    print(f"  {old_top.loc[i,'contentId']:>15} {old_top.loc[i,'popularity_score']:>8.1f}"
          f"  |  {new_top.loc[i,'contentId']:>15} {new_top.loc[i,'popularity_score_new']:>8.1f}")


# ══════════════════════════════════════════
print("\n" + "=" * 60)
print("STEP 6. 조건별 다양성 검증")
print("=" * 60)

def parse_list_col(series):
    def _parse(x):
        try: return ast.literal_eval(x)
        except: return []
    return series.apply(_parse)

def group_tone(tone):
    if pd.isna(tone): return "unknown"
    tone = str(tone)
    if "Gray" in tone or "White" in tone or "Black" in tone: return "무채색"
    elif "Orange" in tone or "Yellow" in tone: return "웜톤"
    else: return "기타"

posts["style_main"] = parse_list_col(posts["features_styleList"]).apply(
    lambda x: x[0] if len(x) > 0 else "unknown")
posts["tone_group"] = posts["primary_tone"].apply(group_tone)

conditions = [
    ("아파트",       "내추럴", "무채색"),
    ("아파트",       "모던",   "무채색"),
    ("아파트",       "내추럴", "웜톤"),
    ("원룸&오피스텔", "내추럴", "무채색"),
    ("원룸&오피스텔", "모던",   "무채색"),
]

print(f"\n  {'조건':<35} {'겹치는 게시글':>12}  {'해석'}")
print("  " + "-" * 70)

overlap_results = []
for residence, style, tone in conditions:
    sub = posts[
        (posts["features_residence"] == residence) &
        (posts["style_main"] == style) &
        (posts["tone_group"] == tone)
    ]
    if len(sub) < 5:
        continue

    old_ids = set(sub.nlargest(5, "popularity_score")["contentId"].tolist())
    new_ids = set(sub.nlargest(5, "popularity_score_new")["contentId"].tolist())
    overlap = len(old_ids & new_ids)
    overlap_results.append(overlap)

    interpret = "동일" if overlap == 5 else f"{5-overlap}개 교체"
    print(f"  {residence}+{style}+{tone:<10} {overlap}/5{'':>8}  {interpret}")

avg_overlap = np.mean(overlap_results) if overlap_results else 0
print(f"\n  평균 겹침: {avg_overlap:.1f}/5")
print(f"  → 평균 {5-avg_overlap:.1f}개 상위 게시글 교체됨")
print(f"  → 새 인기도 점수가 조건별로 더 다양한 게시글 발굴")


# ══════════════════════════════════════════
print("\n" + "=" * 60)
print("STEP 7. 최종 가중치 확정")
print("=" * 60)

print(f"\n[엔트로피 + 상관관계 기반 최종 가중치]")
for m, v in sorted(weights_final.items(), key=lambda x: x[1], reverse=True):
    print(f"  {m:<35}: {v*100:.1f}%")

print(f"\n[해석]")
print(f"  scrapRate  — 조회 대비 저장률 (전환율 개념, 구매 의향)")
print(f"  scrapCount — 절대 저장량 (인기 절대량)")
print(f"  shareCount — 바이럴 지수 (트렌드 신호)")
print(f"  likeCount  — 가벼운 반응 (scrapCount와 중복 신호)")
print(f"  commentAndReplyCount — 깊은 관심도")

print(f"\n→ 이 가중치를 STEP 1 전처리 코드에 반영하고 모델 재학습")

STEP 1. 기존 인기도 점수 계산 (비교용)
기존 인기도 점수 계산 완료
count    8000.00
mean       57.03
std        12.05
min         6.78
25%        48.99
50%        57.28
75%        65.18
max       100.00
Name: popularity_score, dtype: float64

STEP 2. 상관관계 분석 — 중복 신호 확인

지표 간 상관계수:
                      viewCount  likeCount  scrapCount  shareCount  \
viewCount                 1.000      0.896       0.891       0.793   
likeCount                 0.896      1.000       0.956       0.827   
scrapCount                0.891      0.956       1.000       0.734   
shareCount                0.793      0.827       0.734       1.000   
commentAndReplyCount      0.607      0.603       0.505       0.601   

                      commentAndReplyCount  
viewCount                            0.607  
likeCount                            0.603  
scrapCount                           0.505  
shareCount                           0.601  
commentAndReplyCount                 1.000  

지표별 평균 상관계수 (높을수록 중복 신호):
  likeCount             

---
## STEP 4-B. KNN K값 튜닝

### 목적
K값에 따른 **조건 민감도**(자카드 유사도)와 **추천 결과 수** 간의 트레이드오프를 분석한다.

### 평가 기준
- 조건 민감도(60%): 자카드 유사도 낮을수록 맞춤화 우수
- 추천 결과 수(40%): 많을수록 선택지 풍부

### 결론
- **K_DEFAULT = 20** (기본값)
- **K_FALLBACK = 50** (상품 부족 시)


In [7]:
# ══════════════════════════════════════════
# KNN K값 튜닝 — 조건 민감도 + 추천 결과 수
# ══════════════════════════════════════════

import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors
import warnings
warnings.filterwarnings("ignore")

BASE = r"C:\Users\SAMSUNG\Desktop\ohouse_final"

# ── 모델 로드 ──
import pickle
with open(f"{BASE}\\recommend_model_final.pkl", "rb") as f:
    model = pickle.load(f)

posts      = model["posts"]
tags_valid = model["tags_valid"]
scaler     = model["scaler"]
feature_cols  = model["feature_cols"]
VALID_PLACES  = model["VALID_PLACES"]
KNN_CAT_COLS  = model["KNN_CAT_COLS"]
KNN_NUM_COLS  = model["KNN_NUM_COLS"]


def build_feature_matrix(df):
    df = df.copy()
    for col in KNN_CAT_COLS:
        df[col] = df[col].fillna("unknown").astype(str)
    encoded = pd.get_dummies(df[KNN_CAT_COLS], prefix=KNN_CAT_COLS)
    num     = df[KNN_NUM_COLS].fillna(0)
    return pd.concat([
        df[["contentId"]].reset_index(drop=True),
        encoded.reset_index(drop=True),
        num.reset_index(drop=True)
    ], axis=1)


def get_candidates_for_k(filtered, user_row, K):
    """K값에 따른 후보 게시글 추출"""
    X_filtered = build_feature_matrix(filtered)
    for col in feature_cols:
        if col not in X_filtered.columns:
            X_filtered[col] = 0
    X_filtered_scaled = scaler.transform(X_filtered[feature_cols].values)

    X_user = build_feature_matrix(user_row)
    for col in feature_cols:
        if col not in X_user.columns:
            X_user[col] = 0
    user_scaled = scaler.transform(X_user[feature_cols].values)

    k_actual = min(K, len(filtered))
    knn = NearestNeighbors(n_neighbors=k_actual, metric="cosine",
                           algorithm="brute", n_jobs=-1)
    knn.fit(X_filtered_scaled)
    _, indices = knn.kneighbors(user_scaled)
    return filtered.iloc[indices[0]]["contentId"].values


def get_recommended_products(candidate_ids, space, depth2, min_count=10):
    """후보 게시글에서 상품 추출"""
    candidate_tags = tags_valid[tags_valid["contentId"].isin(candidate_ids)].copy()
    if space:
        candidate_tags = candidate_tags[candidate_tags["place_label"] == space]
    if depth2:
        sub = candidate_tags[candidate_tags["ohou_category_depth2"] == depth2]
        candidate_tags = sub if len(sub) >= min_count else candidate_tags

    candidate_tags = candidate_tags.merge(
        posts[["contentId", "popularity_score"]], on="contentId", how="left"
    )
    product_score = (
        candidate_tags.groupby(["productId", "productName"])
        .agg(등장수=("contentId","count"), 평균인기도=("popularity_score","mean"))
        .reset_index()
    )
    product_score = product_score[product_score["등장수"] >= 2]
    if len(product_score) == 0:
        product_score = (
            candidate_tags.groupby(["productId", "productName"])
            .agg(등장수=("contentId","count"), 평균인기도=("popularity_score","mean"))
            .reset_index()
        )
    return set(product_score["productId"].tolist())


# ── 테스트 조건 정의 ──
test_conditions = [
    # (residence, area_range, family, agent, expertise, tone, style, space, space_type, depth2)
    ("아파트", "30~40평", "신혼부부", "전문가", "리모델링", "무채색", "내추럴", "거실", "TR", "소파"),
    ("아파트", "30~40평", "신혼부부", "전문가", "리모델링", "웜톤", "빈티지&레트로", "거실", "TR", "소파"),
    ("아파트", "30~40평", "신혼부부", "전문가", "리모델링", "무채색", "모던", "거실", "RT", "소파"),
    ("원룸&오피스텔", "~10평", "싱글라이프", "셀프•DIY", "홈스타일링", "무채색", "모던", "원룸", "RT", "소파"),
    ("아파트", "20~30평", "신혼부부", "셀프•DIY", "홈스타일링", "무채색", "내추럴", "침실", "TH", "침대"),
]

def apply_hard_filter(residence, area_range, family, agent, expertise, space_type, min_posts=10):
    filtered = posts.copy()
    if residence:   filtered = filtered[filtered["features_residence"] == residence]
    if area_range:  filtered = filtered[filtered["area_range"] == area_range]
    if family:      filtered = filtered[filtered["family_main"] == family]
    if agent:       filtered = filtered[filtered["features_agent"] == agent]
    if expertise:   filtered = filtered[filtered["features_expertise"] == expertise]
    if space_type:  filtered = filtered[filtered["space_type"] == space_type]

    if len(filtered) < min_posts:
        filtered = posts.copy()
        if residence:  filtered = filtered[filtered["features_residence"] == residence]
        if area_range: filtered = filtered[filtered["area_range"] == area_range]
        if family:     filtered = filtered[filtered["family_main"] == family]
        if agent:      filtered = filtered[filtered["features_agent"] == agent]
        if space_type: filtered = filtered[filtered["space_type"] == space_type]
    if len(filtered) < min_posts:
        filtered = posts.copy()
        if residence:  filtered = filtered[filtered["features_residence"] == residence]
        if area_range: filtered = filtered[filtered["area_range"] == area_range]
        if family:     filtered = filtered[filtered["family_main"] == family]
        if space_type: filtered = filtered[filtered["space_type"] == space_type]
    if len(filtered) < min_posts:
        filtered = posts.copy()
        if residence:  filtered = filtered[filtered["features_residence"] == residence]
        if area_range: filtered = filtered[filtered["area_range"] == area_range]
        if family:     filtered = filtered[filtered["family_main"] == family]
    if len(filtered) < min_posts:
        filtered = posts.copy()
    return filtered


# ══════════════════════════════════════════
# K값별 평가
# ══════════════════════════════════════════
K_candidates = [10, 20, 30, 50, 70, 100]

print("=" * 70)
print("KNN K값 튜닝 — 조건 민감도 + 추천 결과 수")
print("=" * 70)

results = []

for K in K_candidates:
    sensitivity_scores = []  # 조건 간 자카드 유사도 (낮을수록 맞춤화 좋음)
    result_counts      = []  # 추천 결과 수 (높을수록 좋음)

    # 조건별 추천 상품 집합 수집
    product_sets = []
    for cond in test_conditions:
        residence, area_range, family, agent, expertise, tone, style, space, space_type, depth2 = cond

        filtered = apply_hard_filter(residence, area_range, family, agent, expertise, space_type)

        user_row = filtered.iloc[[0]].copy()
        user_row["style_main"] = style
        user_row["tone_group"] = tone
        user_row["space_type"] = space_type
        user_row[f"has_{space}"] = 1

        candidate_ids = get_candidates_for_k(filtered, user_row, K)
        products = get_recommended_products(candidate_ids, space, depth2)

        product_sets.append(products)
        result_counts.append(len(products))

    # 조건 간 자카드 유사도 계산
    jaccard_scores = []
    for i in range(len(product_sets)):
        for j in range(i+1, len(product_sets)):
            a, b = product_sets[i], product_sets[j]
            if len(a | b) == 0:
                continue
            jaccard = len(a & b) / len(a | b)
            jaccard_scores.append(jaccard)

    avg_jaccard     = np.mean(jaccard_scores) if jaccard_scores else 0
    avg_result_count = np.mean(result_counts)

    results.append({
        "K":              K,
        "avg_jaccard":    avg_jaccard,
        "avg_result_count": avg_result_count,
        "min_result_count": min(result_counts),
    })

    print(f"\n  K={K:>3}")
    print(f"    조건 간 평균 자카드 유사도: {avg_jaccard:.3f} (낮을수록 맞춤화 좋음)")
    print(f"    평균 추천 상품 수:         {avg_result_count:.1f}개")
    print(f"    최소 추천 상품 수:         {min(result_counts)}개")


# ══════════════════════════════════════════
# 최적 K 선택
# ══════════════════════════════════════════
print("\n" + "=" * 70)
print("최적 K 선택")
print("=" * 70)

results_df = pd.DataFrame(results)

# 정규화
results_df["jaccard_norm"] = 1 - (
    (results_df["avg_jaccard"] - results_df["avg_jaccard"].min()) /
    (results_df["avg_jaccard"].max() - results_df["avg_jaccard"].min() + 1e-9)
)
results_df["count_norm"] = (
    (results_df["avg_result_count"] - results_df["avg_result_count"].min()) /
    (results_df["avg_result_count"].max() - results_df["avg_result_count"].min() + 1e-9)
)

# 종합 점수 = 조건 민감도(60%) + 추천 결과 수(40%)
results_df["total_score"] = (
    results_df["jaccard_norm"] * 0.6 +
    results_df["count_norm"]   * 0.4
)

print(f"\n  {'K':>5} {'자카드':>10} {'평균결과수':>10} {'최소결과수':>10} {'종합점수':>10}")
print("  " + "-" * 55)
for _, row in results_df.iterrows():
    print(f"  {int(row['K']):>5} {row['avg_jaccard']:>10.3f} "
          f"{row['avg_result_count']:>10.1f} "
          f"{int(row['min_result_count']):>10} "
          f"{row['total_score']:>10.3f}")

best_k = int(results_df.loc[results_df["total_score"].idxmax(), "K"])
print(f"\n  → 최적 K값: {best_k}")
print(f"  → 조건 민감도(60%) + 추천 결과 수(40%) 기준")

KNN K값 튜닝 — 조건 민감도 + 추천 결과 수

  K= 10
    조건 간 평균 자카드 유사도: 0.029 (낮을수록 맞춤화 좋음)
    평균 추천 상품 수:         20.8개
    최소 추천 상품 수:         4개

  K= 20
    조건 간 평균 자카드 유사도: 0.011 (낮을수록 맞춤화 좋음)
    평균 추천 상품 수:         9.6개
    최소 추천 상품 수:         3개

  K= 30
    조건 간 평균 자카드 유사도: 0.024 (낮을수록 맞춤화 좋음)
    평균 추천 상품 수:         15.6개
    최소 추천 상품 수:         4개

  K= 50
    조건 간 평균 자카드 유사도: 0.041 (낮을수록 맞춤화 좋음)
    평균 추천 상품 수:         27.0개
    최소 추천 상품 수:         10개

  K= 70
    조건 간 평균 자카드 유사도: 0.061 (낮을수록 맞춤화 좋음)
    평균 추천 상품 수:         40.2개
    최소 추천 상품 수:         17개

  K=100
    조건 간 평균 자카드 유사도: 0.082 (낮을수록 맞춤화 좋음)
    평균 추천 상품 수:         51.6개
    최소 추천 상품 수:         22개

최적 K 선택

      K        자카드      평균결과수      최소결과수       종합점수
  -------------------------------------------------------
     10      0.029       20.8          4      0.556
     20      0.011        9.6          3      0.600
     30      0.024       15.6          4      0.543
     50      0.041       27.0         10      0.507

---
## STEP 4-C. 최종 추천 점수 가중치 튜닝

### 목적
상품 랭킹 점수에서 **등장빈도**와 **평균인기도** 간의 최적 비율을 결정한다.

### 평가 기준
- 맞춤화(50%): 조건 간 자카드 유사도
- 브랜드 다양성(30%)
- 추천 결과 수(20%)

### 결론
- **FREQ_W = 0.40** (등장빈도)
- **POP_W = 0.60** (평균인기도)


In [8]:
# ══════════════════════════════════════════
# 최종 추천 점수 가중치 튜닝
# 등장빈도 vs 평균인기도
# ══════════════════════════════════════════

import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors
import pickle
import warnings
warnings.filterwarnings("ignore")

BASE = r"C:\Users\SAMSUNG\Desktop\ohouse_final"

with open(f"{BASE}\\recommend_model_final.pkl", "rb") as f:
    model = pickle.load(f)

posts      = model["posts"]
tags_valid = model["tags_valid"]
scaler     = model["scaler"]
feature_cols  = model["feature_cols"]
VALID_PLACES  = model["VALID_PLACES"]
KNN_CAT_COLS  = model["KNN_CAT_COLS"]
KNN_NUM_COLS  = model["KNN_NUM_COLS"]


def build_feature_matrix(df):
    df = df.copy()
    for col in KNN_CAT_COLS:
        df[col] = df[col].fillna("unknown").astype(str)
    encoded = pd.get_dummies(df[KNN_CAT_COLS], prefix=KNN_CAT_COLS)
    num     = df[KNN_NUM_COLS].fillna(0)
    return pd.concat([
        df[["contentId"]].reset_index(drop=True),
        encoded.reset_index(drop=True),
        num.reset_index(drop=True)
    ], axis=1)


def get_candidates(filtered, user_row, K=20):
    X_filtered = build_feature_matrix(filtered)
    for col in feature_cols:
        if col not in X_filtered.columns:
            X_filtered[col] = 0
    X_filtered_scaled = scaler.transform(X_filtered[feature_cols].values)

    X_user = build_feature_matrix(user_row)
    for col in feature_cols:
        if col not in X_user.columns:
            X_user[col] = 0
    user_scaled = scaler.transform(X_user[feature_cols].values)

    k_actual = min(K, len(filtered))
    knn = NearestNeighbors(n_neighbors=k_actual, metric="cosine",
                           algorithm="brute", n_jobs=-1)
    knn.fit(X_filtered_scaled)
    _, indices = knn.kneighbors(user_scaled)
    return filtered.iloc[indices[0]]["contentId"].values


def get_products_with_weight(candidate_ids, space, depth2,
                              freq_w, pop_w, min_count=10):
    """가중치별 추천 상품 추출"""
    candidate_tags = tags_valid[tags_valid["contentId"].isin(candidate_ids)].copy()
    if space:
        candidate_tags = candidate_tags[candidate_tags["place_label"] == space]
    if depth2:
        sub = candidate_tags[candidate_tags["ohou_category_depth2"] == depth2]
        candidate_tags = sub if len(sub) >= min_count else candidate_tags

    candidate_tags = candidate_tags.merge(
        posts[["contentId", "popularity_score"]], on="contentId", how="left"
    )
    product_score = (
        candidate_tags.groupby(["productId", "productName", "brand",
                                 "sellingPrice", "ohou_category_depth2"])
        .agg(등장수=("contentId","count"), 평균인기도=("popularity_score","mean"))
        .reset_index()
    )
    product_score = product_score[product_score["등장수"] >= 2]
    if len(product_score) == 0:
        product_score = (
            candidate_tags.groupby(["productId", "productName", "brand",
                                     "sellingPrice", "ohou_category_depth2"])
            .agg(등장수=("contentId","count"), 평균인기도=("popularity_score","mean"))
            .reset_index()
        )

    if len(product_score) == 0:
        return pd.DataFrame(), set()

    max_log = np.log1p(product_score["등장수"]).max()
    max_pop = product_score["평균인기도"].max()

    if max_log > 0 and max_pop > 0:
        product_score["최종점수"] = (
            np.log1p(product_score["등장수"]) / max_log * freq_w +
            product_score["평균인기도"] / max_pop * pop_w
        )
    else:
        product_score["최종점수"] = 0

    result = product_score.sort_values("최종점수", ascending=False).head(10)
    return result, set(result["productId"].tolist())


def apply_hard_filter(residence, area_range, family, agent,
                      expertise, space_type, min_posts=10):
    filtered = posts.copy()
    if residence:  filtered = filtered[filtered["features_residence"] == residence]
    if area_range: filtered = filtered[filtered["area_range"] == area_range]
    if family:     filtered = filtered[filtered["family_main"] == family]
    if agent:      filtered = filtered[filtered["features_agent"] == agent]
    if expertise:  filtered = filtered[filtered["features_expertise"] == expertise]
    if space_type: filtered = filtered[filtered["space_type"] == space_type]

    if len(filtered) < min_posts:
        filtered = posts.copy()
        if residence:  filtered = filtered[filtered["features_residence"] == residence]
        if area_range: filtered = filtered[filtered["area_range"] == area_range]
        if family:     filtered = filtered[filtered["family_main"] == family]
        if agent:      filtered = filtered[filtered["features_agent"] == agent]
        if space_type: filtered = filtered[filtered["space_type"] == space_type]
    if len(filtered) < min_posts:
        filtered = posts.copy()
        if residence:  filtered = filtered[filtered["features_residence"] == residence]
        if area_range: filtered = filtered[filtered["area_range"] == area_range]
        if family:     filtered = filtered[filtered["family_main"] == family]
        if space_type: filtered = filtered[filtered["space_type"] == space_type]
    if len(filtered) < min_posts:
        filtered = posts.copy()
        if residence:  filtered = filtered[filtered["features_residence"] == residence]
        if area_range: filtered = filtered[filtered["area_range"] == area_range]
        if family:     filtered = filtered[filtered["family_main"] == family]
    if len(filtered) < min_posts:
        filtered = posts.copy()
    return filtered


# ── 테스트 조건 ──
test_conditions = [
    ("아파트", "30~40평", "신혼부부", "전문가", "리모델링", "무채색", "내추럴", "거실", "TR", "소파"),
    ("아파트", "30~40평", "신혼부부", "전문가", "리모델링", "웜톤", "빈티지&레트로", "거실", "TR", "소파"),
    ("아파트", "30~40평", "신혼부부", "전문가", "리모델링", "무채색", "모던", "거실", "RT", "소파"),
    ("원룸&오피스텔", "~10평", "싱글라이프", "셀프•DIY", "홈스타일링", "무채색", "모던", "원룸", "RT", "소파"),
    ("아파트", "20~30평", "신혼부부", "셀프•DIY", "홈스타일링", "무채색", "내추럴", "침실", "TH", "침대"),
]

# 후보 게시글 미리 추출 (K=20, 부족하면 K=50)
print("후보 게시글 추출 중...")
all_candidates = []
for cond in test_conditions:
    residence, area_range, family, agent, expertise, tone, style, space, space_type, depth2 = cond
    filtered = apply_hard_filter(residence, area_range, family, agent, expertise, space_type)

    user_row = filtered.iloc[[0]].copy()
    user_row["style_main"] = style
    user_row["tone_group"] = tone
    user_row["space_type"] = space_type
    user_row[f"has_{space}"] = 1

    # K=20 먼저, 부족하면 K=50
    candidate_ids = get_candidates(filtered, user_row, K=20)
    test_tags = tags_valid[tags_valid["contentId"].isin(candidate_ids)]
    if space:
        test_tags = test_tags[test_tags["place_label"] == space]
    if len(test_tags) < 5:
        candidate_ids = get_candidates(filtered, user_row, K=50)

    all_candidates.append(candidate_ids)
    print(f"  {residence}+{style}+{space_type}: {len(candidate_ids)}건")


# ══════════════════════════════════════════
# 가중치 조합별 평가
# ══════════════════════════════════════════
weight_candidates = [
    (0.7, 0.3),
    (0.6, 0.4),
    (0.5, 0.5),
    (0.4, 0.6),
    (0.3, 0.7),
]

print("\n" + "=" * 70)
print("최종 추천 점수 가중치 튜닝")
print("=" * 70)

results = []

for freq_w, pop_w in weight_candidates:
    jaccard_scores = []
    result_counts  = []
    diversity_scores = []  # 추천 결과 내 브랜드 다양성

    product_sets = []
    for i, cond in enumerate(test_conditions):
        residence, area_range, family, agent, expertise, tone, style, space, space_type, depth2 = cond
        candidate_ids = all_candidates[i]

        result, prod_set = get_products_with_weight(
            candidate_ids, space, depth2, freq_w, pop_w
        )
        product_sets.append(prod_set)
        result_counts.append(len(prod_set))

        # 브랜드 다양성
        if not result.empty and "brand" in result.columns:
            n_brands = result["brand"].nunique()
            diversity_scores.append(n_brands / max(len(result), 1))

    # 조건 간 자카드 유사도
    for i in range(len(product_sets)):
        for j in range(i+1, len(product_sets)):
            a, b = product_sets[i], product_sets[j]
            if len(a | b) == 0:
                continue
            jaccard_scores.append(len(a & b) / len(a | b))

    avg_jaccard   = np.mean(jaccard_scores) if jaccard_scores else 0
    avg_count     = np.mean(result_counts)
    avg_diversity = np.mean(diversity_scores) if diversity_scores else 0

    results.append({
        "freq_w":       freq_w,
        "pop_w":        pop_w,
        "avg_jaccard":  avg_jaccard,
        "avg_count":    avg_count,
        "avg_diversity": avg_diversity,
    })

    print(f"\n  등장빈도({freq_w*100:.0f}%) + 인기도({pop_w*100:.0f}%)")
    print(f"    조건 간 자카드 유사도: {avg_jaccard:.3f} (낮을수록 맞춤화 좋음)")
    print(f"    평균 추천 상품 수:    {avg_count:.1f}개")
    print(f"    브랜드 다양성:        {avg_diversity:.3f} (높을수록 다양)")


# ══════════════════════════════════════════
# 최적 가중치 선택
# ══════════════════════════════════════════
print("\n" + "=" * 70)
print("최적 가중치 선택")
print("=" * 70)

results_df = pd.DataFrame(results)

# 정규화
results_df["jaccard_norm"] = 1 - (
    (results_df["avg_jaccard"] - results_df["avg_jaccard"].min()) /
    (results_df["avg_jaccard"].max() - results_df["avg_jaccard"].min() + 1e-9)
)
results_df["count_norm"] = (
    (results_df["avg_count"] - results_df["avg_count"].min()) /
    (results_df["avg_count"].max() - results_df["avg_count"].min() + 1e-9)
)
results_df["diversity_norm"] = (
    (results_df["avg_diversity"] - results_df["avg_diversity"].min()) /
    (results_df["avg_diversity"].max() - results_df["avg_diversity"].min() + 1e-9)
)

# 종합 점수 = 맞춤화(50%) + 다양성(30%) + 결과수(20%)
results_df["total_score"] = (
    results_df["jaccard_norm"]   * 0.5 +
    results_df["diversity_norm"] * 0.3 +
    results_df["count_norm"]     * 0.2
)

print(f"\n  {'가중치':<20} {'자카드':>8} {'결과수':>8} {'다양성':>8} {'종합점수':>10}")
print("  " + "-" * 60)
for _, row in results_df.iterrows():
    label = f"빈도{row['freq_w']*100:.0f}%+인기{row['pop_w']*100:.0f}%"
    print(f"  {label:<20} {row['avg_jaccard']:>8.3f} "
          f"{row['avg_count']:>8.1f} "
          f"{row['avg_diversity']:>8.3f} "
          f"{row['total_score']:>10.3f}")

best_idx = results_df["total_score"].idxmax()
best_freq = results_df.loc[best_idx, "freq_w"]
best_pop  = results_df.loc[best_idx, "pop_w"]

print(f"\n  → 최적 가중치: 등장빈도({best_freq*100:.0f}%) + 인기도({best_pop*100:.0f}%)")
print(f"  → 맞춤화(50%) + 브랜드다양성(30%) + 결과수(20%) 기준")

후보 게시글 추출 중...
  아파트+내추럴+TR: 20건
  아파트+빈티지&레트로+TR: 20건
  아파트+모던+RT: 20건
  원룸&오피스텔+모던+RT: 20건
  아파트+내추럴+TH: 20건

최종 추천 점수 가중치 튜닝

  등장빈도(70%) + 인기도(30%)
    조건 간 자카드 유사도: 0.013 (낮을수록 맞춤화 좋음)
    평균 추천 상품 수:    7.4개
    브랜드 다양성:        0.767 (높을수록 다양)

  등장빈도(60%) + 인기도(40%)
    조건 간 자카드 유사도: 0.013 (낮을수록 맞춤화 좋음)
    평균 추천 상품 수:    7.4개
    브랜드 다양성:        0.767 (높을수록 다양)

  등장빈도(50%) + 인기도(50%)
    조건 간 자카드 유사도: 0.006 (낮을수록 맞춤화 좋음)
    평균 추천 상품 수:    7.4개
    브랜드 다양성:        0.767 (높을수록 다양)

  등장빈도(40%) + 인기도(60%)
    조건 간 자카드 유사도: 0.006 (낮을수록 맞춤화 좋음)
    평균 추천 상품 수:    7.4개
    브랜드 다양성:        0.787 (높을수록 다양)

  등장빈도(30%) + 인기도(70%)
    조건 간 자카드 유사도: 0.006 (낮을수록 맞춤화 좋음)
    평균 추천 상품 수:    7.4개
    브랜드 다양성:        0.787 (높을수록 다양)

최적 가중치 선택

  가중치                       자카드      결과수      다양성       종합점수
  ------------------------------------------------------------
  빈도70%+인기30%             0.013      7.4    0.767      0.000
  빈도60%+인기40%             0.013      7.4    0.767      0.000
  빈도

---
## STEP 1~5. 최종 모델 파이프라인

위 튜닝 결과를 반영한 최종 파이프라인.
**STEP 1 → STEP 3 → STEP 5** 순서로 순차 실행한다.

| 셀 | 내용 | 출력 파일 |
|---|---|---|
| STEP 1 | 전처리 + 피처 테이블 생성 | `model_feature_table.csv`, `model_posts_meta.csv` |
| STEP 3 | KNN 모델 학습 + 추천 함수 정의 + 테스트 | (메모리) |
| STEP 5 | grid_zone 집계 + 최종 모델 저장 | `recommend_model_final.pkl` 외 3개 CSV |


In [9]:
# ══════════════════════════════════════════
# STEP 1. 데이터 로드 + 전처리 + 피처 테이블 생성 (최종)
# ══════════════════════════════════════════

import pandas as pd
import numpy as np
import ast
import warnings
warnings.filterwarnings("ignore")

BASE = r"C:\Users\SAMSUNG\Desktop\ohouse_final"

print("로딩 중...")
posts    = pd.read_csv(f"{BASE}\\housewarming_posts_with_tone_urlinfo.csv")
tags     = pd.read_csv(f"{BASE}\\housewarming_product_tags_with_image.csv", low_memory=False)
space_df = pd.read_csv(f"{BASE}\\housewarming_posts_with_space_type.csv",
                       usecols=["contentId", "space_type"])

print(f"게시글: {posts.shape[0]:,}행 / 태그: {tags.shape[0]:,}행")


# ── 전처리 ──
def parse_list_col(series):
    def _parse(x):
        try: return ast.literal_eval(x)
        except: return []
    return series.apply(_parse)

def group_tone(tone):
    if pd.isna(tone): return None
    tone = str(tone)
    if "Gray" in tone or "White" in tone or "Black" in tone: return "무채색"
    elif "Orange" in tone or "Yellow" in tone: return "웜톤"
    elif "Blue" in tone or "Sky" in tone: return "쿨톤"
    elif "Green" in tone: return "그린톤"
    elif "Red" in tone or "Pink" in tone: return "레드·핑크톤"
    else: return "기타"

posts["style_main"]        = parse_list_col(posts["features_styleList"]).apply(
    lambda x: x[0] if len(x) > 0 else None)
posts["family_main"]       = parse_list_col(posts["features_familyList"]).apply(
    lambda x: x[0] if len(x) > 0 else None)
posts["construction_main"] = parse_list_col(posts["features_constructions"]).apply(
    lambda x: x[0] if len(x) > 0 else None)
posts["area_num"]    = posts["features_area"].str.extract(r"(\d+)").astype(float)
posts["tone_group"]  = posts["primary_tone"].apply(group_tone)
posts["tone2_group"] = posts["secondary_tone"].apply(group_tone)
posts["sido"]        = posts["features_region"].str.extract(
    r"^([\w]+(?:특별시|광역시|도|특별자치도|특별자치시)?)")

area_bins   = [0, 10, 20, 30, 40, 50, np.inf]
area_labels = ["~10평", "10~20평", "20~30평", "30~40평", "40~50평", "50평+"]
posts["area_range"] = pd.cut(posts["area_num"], bins=area_bins, labels=area_labels)

# space_type 병합
posts = posts.merge(space_df, on="contentId", how="left")
posts["space_type"] = posts["space_type"].fillna("unknown")

# ── 인기도 점수 (엔트로피 + 상관관계 기반 가중치) ──
posts["scrapRate"] = posts["scrapCount"] / (posts["viewCount"] + 1)

METRICS_FINAL = ["scrapRate", "scrapCount", "likeCount", "shareCount", "commentAndReplyCount"]
WEIGHTS_FINAL = {
    "scrapRate":              0.312,
    "commentAndReplyCount":   0.249,
    "shareCount":             0.184,
    "likeCount":              0.130,
    "scrapCount":             0.125,
}

for m in METRICS_FINAL:
    posts[f"{m}_log"]  = np.log1p(posts[m].fillna(0))
    max_val = posts[f"{m}_log"].max()
    posts[f"{m}_norm"] = posts[f"{m}_log"] / max_val if max_val > 0 else 0

posts["popularity_score"] = sum(
    posts[f"{m}_norm"] * WEIGHTS_FINAL[m] for m in METRICS_FINAL
)
posts["popularity_score"] = (
    posts["popularity_score"] / posts["popularity_score"].max() * 100
).round(2)

print(f"인기도 점수 재설계 완료 (엔트로피+상관관계 기반)")
print(f"  중앙값: {posts['popularity_score'].median():.2f}")

# ── 태그 집계 ──
print("태그 집계 중...")
tags_valid = tags.dropna(subset=["productId", "sellingPrice", "ohou_category_depth1"])
tags_valid = tags_valid[tags_valid["sellingPrice"] > 0]
tags_dedup = tags_valid.drop_duplicates(subset=["contentId", "productId"])

VALID_PLACES = [
    "거실", "주방", "침실", "아이방", "원룸",
    "서재&작업실", "욕실", "베란다", "드레스룸",
    "사무공간", "현관", "복도", "가구&소품"
]

tags_place = tags_dedup[tags_dedup["place_label"].isin(VALID_PLACES)]
tags_place_dedup = tags_place.drop_duplicates(
    subset=["contentId", "place_label", "productId"])
place_budget = (
    tags_place_dedup.groupby(["contentId", "place_label"])["sellingPrice"]
    .sum().reset_index()
)

for place in VALID_PLACES:
    sub = place_budget[place_budget["place_label"] == place][["contentId", "sellingPrice"]]
    posts = posts.merge(
        sub.rename(columns={"sellingPrice": f"budget_{place}"}),
        on="contentId", how="left"
    )
    place_posts = set(tags_place[tags_place["place_label"] == place]["contentId"].unique())
    posts[f"has_{place}"] = posts["contentId"].isin(place_posts).astype(int)

print(f"태그 집계 완료")
print(f"게시글 전처리 완료: {posts.shape[1]}컬럼")

로딩 중...
게시글: 8,000행 / 태그: 1,941,343행
인기도 점수 재설계 완료 (엔트로피+상관관계 기반)
  중앙값: 53.27
태그 집계 중...
태그 집계 완료
게시글 전처리 완료: 73컬럼


In [10]:
# ══════════════════════════════════════════
# STEP 3. KNN 추천 모델 학습 + 저장 (최종)
# ══════════════════════════════════════════

from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
import pickle

KNN_CAT_COLS = ["style_main", "tone_group", "tone2_group", "space_type"]
KNN_NUM_COLS = (
    ["primary_ratio"] +
    [f"budget_{p}" for p in VALID_PLACES] +
    [f"has_{p}" for p in VALID_PLACES]
)

def build_feature_matrix(df):
    df = df.copy()
    for col in KNN_CAT_COLS:
        df[col] = df[col].fillna("unknown").astype(str)
    encoded = pd.get_dummies(df[KNN_CAT_COLS], prefix=KNN_CAT_COLS)
    num     = df[KNN_NUM_COLS].fillna(0)
    return pd.concat([
        df[["contentId"]].reset_index(drop=True),
        encoded.reset_index(drop=True),
        num.reset_index(drop=True)
    ], axis=1), list(encoded.columns)

X_full, encoded_cols = build_feature_matrix(posts)
feature_cols = encoded_cols + KNN_NUM_COLS

scaler = StandardScaler()
scaler.fit(X_full[feature_cols].values)

print(f"KNN 피처 수: {len(feature_cols)}개")
print("스케일러 학습 완료")


# ── 추천 함수 (최종 튜닝값 반영) ──
# 튜닝 확정값
K_DEFAULT   = 20    # 기본 K값
K_FALLBACK  = 50    # 상품 부족 시 확장 K값
MIN_RESULT  = 5     # 최소 추천 결과 수
MIN_COUNT   = 10    # Fallback 최소 기준
FREQ_W      = 0.4   # 등장빈도 가중치
POP_W       = 0.6   # 인기도 가중치

def recommend(
    residence=None, area_range=None, family=None,
    agent=None, expertise=None,
    tone=None, style=None, space=None, budget=None,
    space_type=None,
    change_type="방전체",
    depth1=None, depth2=None, depth3=None, depth4=None,
    top_n=10
):
    # ── Step 1. 하드 필터 ──
    filtered = posts.copy()
    if residence:  filtered = filtered[filtered["features_residence"] == residence]
    if area_range: filtered = filtered[filtered["area_range"] == area_range]
    if family:     filtered = filtered[filtered["family_main"] == family]
    if agent:      filtered = filtered[filtered["features_agent"] == agent]
    if expertise:  filtered = filtered[filtered["features_expertise"] == expertise]
    if space_type: filtered = filtered[filtered["space_type"] == space_type]
    if budget and space:
        budget_col = f"budget_{space}"
        if budget_col in filtered.columns:
            filtered = filtered[
                filtered[budget_col].isna() | (filtered[budget_col] <= budget)
            ]

    # 단계적 완화
    if len(filtered) < 10:
        print(f"  ⚠️ expertise 완화 ({len(filtered)}건 → ", end="")
        filtered = posts.copy()
        if residence:  filtered = filtered[filtered["features_residence"] == residence]
        if area_range: filtered = filtered[filtered["area_range"] == area_range]
        if family:     filtered = filtered[filtered["family_main"] == family]
        if agent:      filtered = filtered[filtered["features_agent"] == agent]
        if space_type: filtered = filtered[filtered["space_type"] == space_type]
        print(f"{len(filtered)}건)")

    if len(filtered) < 10:
        print(f"  ⚠️ agent 완화 ({len(filtered)}건 → ", end="")
        filtered = posts.copy()
        if residence:  filtered = filtered[filtered["features_residence"] == residence]
        if area_range: filtered = filtered[filtered["area_range"] == area_range]
        if family:     filtered = filtered[filtered["family_main"] == family]
        if space_type: filtered = filtered[filtered["space_type"] == space_type]
        print(f"{len(filtered)}건)")

    if len(filtered) < 10:
        print(f"  ⚠️ space_type 완화 ({len(filtered)}건 → ", end="")
        filtered = posts.copy()
        if residence:  filtered = filtered[filtered["features_residence"] == residence]
        if area_range: filtered = filtered[filtered["area_range"] == area_range]
        if family:     filtered = filtered[filtered["family_main"] == family]
        print(f"{len(filtered)}건)")

    if len(filtered) < 10:
        print(f"  ⚠️ 전체 사용")
        filtered = posts.copy()

    print(f"  하드 필터 후: {len(filtered):,}건")

    # ── Step 2. KNN (K=20, 부족하면 K=50) ──
    def run_knn(filtered, user_row, K):
        X_filtered, _ = build_feature_matrix(filtered)
        for col in feature_cols:
            if col not in X_filtered.columns:
                X_filtered[col] = 0
        X_filtered_scaled = scaler.transform(X_filtered[feature_cols].values)

        X_user, _ = build_feature_matrix(user_row)
        for col in feature_cols:
            if col not in X_user.columns:
                X_user[col] = 0
        user_scaled = scaler.transform(X_user[feature_cols].values)

        k_actual = min(K, len(filtered))
        knn = NearestNeighbors(n_neighbors=k_actual, metric="cosine",
                               algorithm="brute", n_jobs=-1)
        knn.fit(X_filtered_scaled)
        _, indices = knn.kneighbors(user_scaled)
        return filtered.iloc[indices[0]]["contentId"].values

    user_row = filtered.iloc[[0]].copy()
    if style:      user_row["style_main"]  = style
    if tone:       user_row["tone_group"]  = tone
    if space_type: user_row["space_type"]  = space_type
    if budget and space: user_row[f"budget_{space}"] = budget
    if space:      user_row[f"has_{space}"] = 1

    # K=20 먼저
    candidate_ids = run_knn(filtered, user_row, K_DEFAULT)

    # 상품 부족 확인 → K=50으로 확장
    check_tags = tags_valid[tags_valid["contentId"].isin(candidate_ids)]
    if space:
        check_tags = check_tags[check_tags["place_label"] == space]
    if len(check_tags) < MIN_RESULT:
        print(f"  ⚠️ 상품 부족 → K={K_FALLBACK}으로 확장")
        candidate_ids = run_knn(filtered, user_row, K_FALLBACK)

    print(f"  KNN 후보: {len(candidate_ids):,}건")

    # ── Step 3. 상품 추출 + Fallback ──
    candidate_tags = tags_valid[tags_valid["contentId"].isin(candidate_ids)].copy()

    if change_type in ["특정공간", "단일상품"] and space:
        candidate_tags = candidate_tags[candidate_tags["place_label"] == space]

    def filter_with_fallback(df, d1, d2, d3, d4, min_count=MIN_COUNT):
        for depth_col, val in [
            ("ohou_category_depth4", d4),
            ("ohou_category_depth3", d3),
            ("ohou_category_depth2", d2),
            ("ohou_category_depth1", d1),
        ]:
            if val:
                sub = df[df[depth_col] == val]
                if len(sub) >= min_count:
                    print(f"  Fallback: {depth_col} 사용 ({len(sub)}건)")
                    return sub
        print(f"  Fallback: 카테고리 필터 없음 ({len(df)}건)")
        return df

    if change_type in ["단일상품", "특정공간"] and any([depth1, depth2, depth3, depth4]):
        candidate_tags = filter_with_fallback(
            candidate_tags, depth1, depth2, depth3, depth4
        )

    # ── Step 4. 인기도 랭킹 (등장빈도 40% + 인기도 60%) ──
    candidate_tags = candidate_tags.merge(
        posts[["contentId", "popularity_score"]], on="contentId", how="left"
    )
    product_score = (
        candidate_tags.groupby([
            "productId", "productName", "brand", "sellingPrice",
            "ohou_category_depth1", "ohou_category_depth2",
            "ohou_category_depth3", "originalImageUrl", "productUrl"
        ])
        .agg(등장수=("contentId","count"), 평균인기도=("popularity_score","mean"))
        .reset_index()
    )
    product_score = product_score[product_score["등장수"] >= 2]
    if len(product_score) == 0:
        product_score = (
            candidate_tags.groupby([
                "productId", "productName", "brand", "sellingPrice",
                "ohou_category_depth1", "ohou_category_depth2",
                "ohou_category_depth3", "originalImageUrl", "productUrl"
            ])
            .agg(등장수=("contentId","count"), 평균인기도=("popularity_score","mean"))
            .reset_index()
        )

    if len(product_score) == 0:
        print("  ❌ 추천 결과 없음")
        return pd.DataFrame()

    max_log = np.log1p(product_score["등장수"]).max()
    max_pop = product_score["평균인기도"].max()
    product_score["최종점수"] = (
        np.log1p(product_score["등장수"]) / max_log * FREQ_W +
        product_score["평균인기도"] / max_pop * POP_W
        if max_log > 0 and max_pop > 0 else 0
    )

    result = product_score.sort_values("최종점수", ascending=False).head(top_n)

    print(f"\n  추천 결과 ({len(result)}개):")
    print(f"  {'상품명':<40} {'브랜드':<12} {'가격':>10} {'점수':>6}")
    print("  " + "-" * 72)
    for _, row in result.iterrows():
        print(f"  {str(row['productName'])[:40]:<40} "
              f"{str(row['brand'])[:12]:<12} "
              f"{row['sellingPrice']:>9,.0f}원 "
              f"{row['최종점수']:>5.3f}")

    return result


# ── 테스트 ──
print("\n" + "="*60)
print("[테스트 1] 아파트 + 신혼부부 + 전문가 + TR + 소파")
print("="*60)
result1 = recommend(
    residence="아파트", area_range="30~40평", family="신혼부부",
    agent="전문가", expertise="리모델링",
    tone="무채색", style="내추럴", space_type="TR",
    space="거실", budget=5000000,
    change_type="특정공간", depth1="가구", depth2="소파", top_n=10
)

print("\n" + "="*60)
print("[테스트 2] 원룸 + 싱글 + 셀프DIY + RT + 방전체")
print("="*60)
result2 = recommend(
    residence="원룸&오피스텔", area_range="~10평", family="싱글라이프",
    agent="셀프•DIY", expertise="홈스타일링",
    tone="무채색", style="모던", space_type="RT",
    change_type="방전체", budget=2000000, top_n=10
)

print("\n" + "="*60)
print("[테스트 3] 아파트 + 신혼부부 + 전문가 + HT + 서재 + 책상")
print("="*60)
result3 = recommend(
    residence="아파트", area_range="30~40평", family="신혼부부",
    agent="전문가", expertise="리모델링",
    tone="무채색", style="모던", space_type="HT",
    space="서재&작업실", budget=3000000,
    change_type="특정공간", depth1="가구", depth2="테이블·식탁·책상", top_n=10
)

KNN 피처 수: 66개
스케일러 학습 완료

[테스트 1] 아파트 + 신혼부부 + 전문가 + TR + 소파
  하드 필터 후: 48건
  KNN 후보: 20건
  Fallback: ohou_category_depth2 사용 (64건)

  추천 결과 (10개):
  상품명                                      브랜드                  가격     점수
  ------------------------------------------------------------------------
  [5%쿠폰] 클레멘티 3인+카우치 폴쉬 린넨 워셔블 패브릭소파 2type 버즈가구         1,870,000원 0.888
  브로디 기능성 패브릭 4인용 소파                       디토소파         2,380,000원 0.868
  헨트 3인소파 (2colors)                        까사미아         2,090,000원 0.821
  아쿠아패브릭 론 카우치 소파 3colors                  먼데이하우스         489,000원 0.783
  메시 전동 1인 리클라이너 소파 - 옐로우                  라오메뜨         3,000,000원 0.772
  에반 아쿠아텍스 4인용 카우치 소파 4colors              웰퍼니쳐           449,000원 0.723
  밀크 4인 조야 기능성 패브릭 소파(스툴 추가구매)             도모디자인          499,000원 0.720
  [10%쿠폰]ILANG 아쿠아텍스 3.5인소파 2colors_쿠션증정   먼데이하우스         299,000원 0.696
  우스터 1인 리클라이너 (3colors)                   까사미아           888,250원 0.691
  알마 4인용 가죽소파 2colors                      자네트

In [11]:
# ══════════════════════════════════════════
# STEP 5. grid_zone + 최종 모델 저장
# ══════════════════════════════════════════

print("grid_zone 집계 중...")

tags_grid = tags_valid.dropna(
    subset=["grid_zone", "place_label", "ohou_category_depth2"])
tags_grid = tags_grid[tags_grid["place_label"].isin(VALID_PLACES)]

grid_pattern = (
    tags_grid.groupby(["place_label", "grid_zone", "ohou_category_depth2"])
    .size().reset_index(name="count")
)
grid_total = grid_pattern.groupby(["place_label", "grid_zone"])["count"].transform("sum")
grid_pattern["ratio"] = grid_pattern["count"] / grid_total

grid_top = (
    grid_pattern.sort_values(["place_label", "grid_zone", "ratio"],
                              ascending=[True, True, False])
    .groupby(["place_label", "grid_zone"]).head(5)
    .reset_index(drop=True)
)

grid_representative = (
    grid_pattern.sort_values(["place_label", "grid_zone", "ratio"],
                              ascending=[True, True, False])
    .groupby(["place_label", "grid_zone"]).first()
    .reset_index()
    [["place_label", "grid_zone", "ohou_category_depth2", "ratio"]]
    .rename(columns={"ohou_category_depth2": "top_category"})
)

tags_pos = tags_valid.dropna(subset=[
    "positionX_norm", "positionY_norm",
    "place_label", "ohou_category_depth2"
])
tags_pos = tags_pos[tags_pos["place_label"].isin(VALID_PLACES)]

category_position = (
    tags_pos.groupby(["place_label", "ohou_category_depth2"])
    .agg(
        median_x=("positionX_norm", "median"),
        median_y=("positionY_norm", "median"),
        std_x=("positionX_norm", "std"),
        std_y=("positionY_norm", "std"),
        count=("productId", "count")
    )
    .reset_index()
)
category_position = category_position[category_position["count"] >= 50]
print(f"grid_zone 집계 완료")

# ── 최종 저장 ──
print("\n최종 모델 저장 중...")

final_model = {
    # KNN 구성요소
    "scaler":        scaler,
    "feature_cols":  feature_cols,
    "encoded_cols":  encoded_cols,
    "KNN_CAT_COLS":  KNN_CAT_COLS,
    "KNN_NUM_COLS":  KNN_NUM_COLS,

    # 튜닝 확정값
    "K_DEFAULT":   K_DEFAULT,
    "K_FALLBACK":  K_FALLBACK,
    "MIN_RESULT":  MIN_RESULT,
    "MIN_COUNT":   MIN_COUNT,
    "FREQ_W":      FREQ_W,
    "POP_W":       POP_W,
    "WEIGHTS_FINAL": WEIGHTS_FINAL,

    # 데이터
    "posts":       posts,
    "tags_valid":  tags_valid,
    "VALID_PLACES": VALID_PLACES,

    # 배치 데이터
    "grid_pattern":        grid_pattern,
    "grid_top":            grid_top,
    "grid_representative": grid_representative,
    "category_position":   category_position,
}

with open(f"{BASE}\\recommend_model_final.pkl", "wb") as f:
    pickle.dump(final_model, f)

grid_top.to_csv(f"{BASE}\\grid_zone_pattern.csv",
                index=False, encoding="utf-8-sig")
grid_representative.to_csv(f"{BASE}\\grid_zone_representative.csv",
                           index=False, encoding="utf-8-sig")
category_position.to_csv(f"{BASE}\\category_position.csv",
                         index=False, encoding="utf-8-sig")

print("\n저장 완료:")
print("  recommend_model_final.pkl")
print("  grid_zone_pattern.csv")
print("  grid_zone_representative.csv")
print("  category_position.csv")
print("\n✅ 모델링 완료!")
print("\n[최종 튜닝 확정값]")
print(f"  인기도 가중치: scrapRate({WEIGHTS_FINAL['scrapRate']*100:.1f}%) + "
      f"commentAndReply({WEIGHTS_FINAL['commentAndReplyCount']*100:.1f}%) + "
      f"shareCount({WEIGHTS_FINAL['shareCount']*100:.1f}%) + "
      f"likeCount({WEIGHTS_FINAL['likeCount']*100:.1f}%) + "
      f"scrapCount({WEIGHTS_FINAL['scrapCount']*100:.1f}%)")
print(f"  KNN K값: {K_DEFAULT} (부족 시 {K_FALLBACK})")
print(f"  Fallback 최소 기준: {MIN_COUNT}건")
print(f"  최종 추천 점수: 등장빈도({FREQ_W*100:.0f}%) + 인기도({POP_W*100:.0f}%)")

grid_zone 집계 중...
grid_zone 집계 완료

최종 모델 저장 중...

저장 완료:
  recommend_model_final.pkl
  grid_zone_pattern.csv
  grid_zone_representative.csv
  category_position.csv

✅ 모델링 완료!

[최종 튜닝 확정값]
  인기도 가중치: scrapRate(31.2%) + commentAndReply(24.9%) + shareCount(18.4%) + likeCount(13.0%) + scrapCount(12.5%)
  KNN K값: 20 (부족 시 50)
  Fallback 최소 기준: 10건
  최종 추천 점수: 등장빈도(40%) + 인기도(60%)
